# Claude Certified Architect — Foundations
## Domain 3: Claude Code Configuration & Workflows
**Exam weight: 20%**

Claude Code is a developer tool first, but the decisions an architect makes about its configuration determine whether it scales consistently across a team or quietly diverges per developer. This domain covers the configuration layer — CLAUDE.md hierarchy, commands and skills, path-scoped rules, plan mode, iterative refinement, and CI/CD integration — and the underlying principles that explain why each design decision matters.

This domain is configuration and workflow knowledge — most concepts cannot be demonstrated by calling the Claude API directly because they live in the Claude Code CLI environment. Each section explains the concept, shows the correct config structure or CLI usage, and where possible uses the API to illustrate the underlying principle.

**Prerequisites:** `pip install anthropic`  
**Auth:** Set `ANTHROPIC_API_KEY` as an environment variable.

### Task Statements Covered
- **3.1** Configure CLAUDE.md files with appropriate hierarchy, scoping, and modular organization
- **3.2** Create and configure custom slash commands and skills
- **3.3** Apply path-specific rules for conditional convention loading
- **3.4** Determine when to use plan mode vs direct execution
- **3.5** Apply iterative refinement techniques for progressive improvement
- **3.6** Integrate Claude Code into CI/CD pipelines

In [1]:
import anthropic
import json
import os

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-6"
print("Client ready.")

Client ready.


---
## Task Statement 3.1: Configure CLAUDE.md files with appropriate hierarchy, scoping, and modular organization

Claude Code reads CLAUDE.md files to understand how a project works — its architecture, standards, and non-negotiables — before touching any code. Where those files live determines whether their instructions reach one developer or the whole team, whether they load for every task or only when relevant, and whether the context they add helps or just fills up the token budget. Getting the hierarchy and structure right is the difference between a configuration that scales and one that quietly diverges per developer.

**What this means in practice:** CLAUDE.md files provide persistent context to Claude Code. They load automatically — unlike skills, which are invoked on demand. There are three scoping levels:
- `~/.claude/CLAUDE.md` — user-level: personal preferences, applies only to you, NOT shared via version control
- `.claude/CLAUDE.md` or root `CLAUDE.md` — project-level: team standards, committed to the repo
- Subdirectory `CLAUDE.md` — directory-level: conventions specific to that subtree

The `@import` syntax keeps files modular. The `.claude/rules/` directory is an alternative to a monolithic CLAUDE.md.

**Why it matters for an architect:** Instructions in user-level config don't propagate to teammates. A new team member whose agent behaves differently from everyone else's is usually a scoping bug — their instructions are in `~/.claude/CLAUDE.md` instead of the project-level file. A monolithic CLAUDE.md that loads 5,000 tokens of context for every task wastes the context window; modular organization via `@import` or `.claude/rules/` lets you load only what's relevant.

**Core concepts:**
- Three scoping levels: user (`~/.claude/CLAUDE.md`, personal, NOT version-controlled), project (`.claude/CLAUDE.md`, shared via version control), directory (subdirectory `CLAUDE.md`, subtree-specific)
- `@import` syntax pulls in focused standards files to keep the root CLAUDE.md short; `.claude/rules/` is an alternative modular organization
- `/memory` command shows which CLAUDE.md files are loaded in the current session

**Anti-patterns to avoid:**
- Placing team standards in `~/.claude/CLAUDE.md` — new teammates won't see them
- Monolithic root CLAUDE.md that loads thousands of tokens for every task regardless of what's being edited
- Using a directory-level CLAUDE.md for conventions that span multiple directories (use `.claude/rules/` with glob patterns instead)

| Level | File path | Shared via version control? | Common mistake |
|-------|-----------|------------------------------|----------------|
| User-level | `~/.claude/CLAUDE.md` | No — personal only, never committed | Putting team standards here; new teammates won't see them |
| Project-level | `.claude/CLAUDE.md` (or root `CLAUDE.md`) | Yes — committed to the repo, applies to all developers who clone it | None — this is the correct place for team standards |
| Subdirectory-level | `src/api/CLAUDE.md` (any subdirectory) | Yes — committed, but applies only to files in that directory and below | Using a subdirectory `CLAUDE.md` for conventions that span multiple directories; use `.claude/rules/` with glob patterns instead |


The correct format for a project-level `CLAUDE.md`. Keep the root file short; use `@import` to pull in focused standards files.

```markdown
# Project: Payment Service

## Architecture
FastAPI + PostgreSQL. Entry point: src/main.py.
Repository pattern for all DB access (src/db/repository.py).

## Standards
@import .claude/standards/testing.md
@import .claude/standards/api-conventions.md
@import .claude/standards/error-handling.md

## Non-negotiable
- Never commit credentials or API keys
- All public functions must have type hints
- Run tests before marking a task complete: `pytest tests/ -v`
```

Each imported file covers one topic. Example — `.claude/standards/testing.md`:

```markdown
# Testing Standards
- Use pytest. Test files: tests/<module_name>_test.py
- Mock all external services (never hit real APIs in tests)
- Minimum coverage: 80% for new code
- Use fixtures from tests/conftest.py for database setup
```

The `@import` syntax keeps the root `CLAUDE.md` under ~500 tokens and lets each standards file evolve independently.


### Common CLAUDE.md Scoping Scenarios

---

**Scenario A — Error handling rules not reaching new team members**

*Description:* A new developer joins and their agent doesn't enforce the team's error handling conventions, even though senior developers' agents do.

*Root cause:* The error handling rules live in a senior developer's `~/.claude/CLAUDE.md` (user-level). That file is never committed to version control, so it is invisible to anyone else who clones the repo.

*Fix:* Move the rules to `.claude/CLAUDE.md` and commit. Run `/memory` in a Claude Code session to verify the project-level file loads correctly.

---

**Scenario B — Subdirectory override not loading**

*Description:* A `src/api/CLAUDE.md` file with API-specific conventions was added to the repo, but the agent doesn't apply those conventions when editing files in `src/api/`.

*Root cause:* The path to the subdirectory `CLAUDE.md` may not be nested directly under the project root that Claude Code is tracking, or the file was added but not yet committed and pushed. Claude Code only loads committed project-level and subdirectory `CLAUDE.md` files.

*Fix:* Confirm the file is committed (`git status`). Run `/memory` while editing a file inside `src/api/` and verify the subdirectory `CLAUDE.md` appears in the loaded-files list.


**Key exam facts for 3.1:**
- Three levels: user (`~/.claude/CLAUDE.md`), project (`.claude/CLAUDE.md`), directory (subdirectory `CLAUDE.md`)
- User-level is NOT shared via version control — team standards go in project-level
- `@import` keeps CLAUDE.md modular — imports topic-specific standards files
- `.claude/rules/` directory = alternative to monolithic CLAUDE.md
- `/memory` command verifies which files are loaded in the current session
- New team member seeing different behavior = almost always a user-level vs project-level scoping bug

---
## Task Statement 3.2: Create and configure custom slash commands and skills

Most teams repeat the same Claude Code workflows constantly — security review before merging, test generation for new modules, codebase orientation for new contributors. Slash commands and skills let you encode those workflows once and invoke them on demand, turning multi-step prompting sequences into a single command that every developer on the team can use consistently. The key architectural decision is whether a skill should run in the main session or in an isolated sub-agent — which determines whether its verbose output stays out of the way or accumulates in the context window.

**What this means in practice:** Two mechanisms for extending Claude Code:
- **Slash commands** (`.claude/commands/`) — Markdown files that define reusable prompts. Project-scoped commands live in `.claude/commands/` and are shared via version control. User-scoped live in `~/.claude/commands/`.
- **Skills** (`.claude/skills/`) — More powerful than commands. SKILL.md files support YAML frontmatter with `context: fork` (run in isolated sub-agent), `allowed-tools` (restrict tool access during execution), and `argument-hint` (prompt for parameters).

**Why it matters for an architect:** The `context: fork` option is the key architectural decision. Without it, a verbose skill (e.g., a full codebase analysis) dumps thousands of tokens into the main session's context, crowding out subsequent work. With `context: fork`, the skill runs in an isolated sub-agent and returns only its summary — main session context stays clean.

**Core concepts:**
- Commands (`.claude/commands/`) are reusable Markdown prompt files; project-scoped commands live in `.claude/commands/` and are shared via version control; personal commands live in `~/.claude/commands/`
- Skills (`.claude/skills/<name>/SKILL.md`) add YAML frontmatter: `context: fork` runs in an isolated sub-agent, `allowed-tools` restricts tool access, `argument-hint` prompts for parameters when the skill is invoked without arguments
- `context: fork` is the key architectural decision — the skill's verbose exploration output stays out of the main session context; only the final summary is returned
- CLAUDE.md is always loaded; commands and skills are invoked on demand — they serve different purposes

**Anti-patterns to avoid:**
- Using `context: fork` for focused, low-output skills (unnecessary overhead; fork is for verbose exploration)
- Not using `context: fork` for verbose skills — the full exploration output enters the main session and crowds out subsequent work
- Placing personal skill variants in the project `.claude/skills/` directory — it affects all teammates

### Slash Command File Locations

| Scope | File path | Shared with team? | Invoked as |
|-------|-----------|-------------------|------------|
| Project-shared | `.claude/commands/<name>.md` | Yes — committed to version control | `/<name>` |
| Personal | `~/.claude/commands/<name>.md` | No — personal only | `/<name>` |

---

### Command file format

A command file is plain Markdown. The entire file content becomes the prompt. There is no required frontmatter.

```markdown
Review the current changes for the following criteria:

1. **Security**: SQL injection, XSS, insecure deserialization, hardcoded credentials
2. **Error handling**: All exceptions caught and logged, no silent failures
3. **Test coverage**: New code has corresponding tests

For each issue found, provide:
- File and line number
- Severity: critical | high | medium | low
- Specific description
- Suggested fix

Do not report style preferences or subjective improvements.
```

*Saved as `.claude/commands/review.md` — invoked with `/review`.*

---

**`$ARGUMENTS` variable:** If the command accepts input, include `$ARGUMENTS` in the prompt body. When the developer invokes `/review src/payments/` the value `src/payments/` is substituted wherever `$ARGUMENTS` appears.


### SKILL.md Frontmatter Format

Skills live at `.claude/skills/<name>/SKILL.md`. The YAML frontmatter controls execution behavior.

| Field | Purpose |
|-------|---------|
| `context: fork` | Run in an isolated sub-agent; only the summary returns to the main session |
| `allowed-tools` | Restrict which tools the skill may call (e.g., read-only during analysis) |
| `argument-hint` | Prompt text shown when the skill is invoked without arguments |
| `$ARGUMENTS` | Placeholder in the prompt body; replaced with whatever the developer passes |

---

**Example 1 — with `context: fork` (verbose/exploratory skill)**

```markdown
---
context: fork
allowed-tools: [Read, Grep, Glob]
argument-hint: "<directory to analyze>"
---

# Codebase Analysis

Analyze the codebase in $ARGUMENTS and produce a structured summary:
1. Entry points and main modules
2. Key dependencies and their purposes
3. Architecture patterns in use
4. High-risk areas (complexity, missing tests, TODOs)

Return ONLY the summary. Do not include raw file contents in your response.
```

`context: fork` keeps the verbose exploration output out of the main session. Only the final summary is returned.

---

**Example 2 — without `context: fork` (focused, low-output skill)**

```markdown
---
allowed-tools: [Read, Write, Bash]
argument-hint: "<file to generate tests for>"
---

# Test Generation

Generate pytest tests for $ARGUMENTS:
1. Read the target file
2. Identify all public functions and methods
3. Write tests covering: happy path, edge cases, error conditions
4. Write the test file to tests/<module>_test.py
5. Run: `pytest tests/<module>_test.py -v`
```

No `context: fork` here — the deliverable (the test file) belongs in the main session.


### `argument-hint` in Practice

`argument-hint` is a CLI gate: when a developer invokes a skill without arguments, Claude Code displays the hint text as a prompt and waits for input before the skill runs. Without it, `$ARGUMENTS` substitutes as an empty string and the skill runs against nothing.

---

**What the developer sees at the terminal**

Without `argument-hint` — skill runs immediately with `$ARGUMENTS = ""`:

```
> /analyze-codebase
[skill runs — $ARGUMENTS is empty, output is vague or nonsensical]
```

With `argument-hint: "<directory to analyze>"` — Claude Code pauses and prompts:

```
> /analyze-codebase
directory to analyze> src/payments/
[skill runs with $ARGUMENTS = "src/payments/"]
```

The hint text is shown verbatim as the prompt label. Write it as a usage hint, not a question — `<file to generate tests for>` not `Which file?`.

---

**What changes in the prompt**

The same `$ARGUMENTS` placeholder appears in both cases. The difference is what gets substituted:

```markdown
# Without argument-hint (or invoked with arguments directly)
Analyze the codebase in src/payments/ and return a structured summary...
                        ^^^^^^^^^^^^^^  ← substituted from developer input or CLI argument

# Without argument-hint, invoked bare
Analyze the codebase in  and return a structured summary...
                        ^  ← empty string — model has no target
```

---

**Skill with `argument-hint` (correct pattern)**

```markdown
---
context: fork
allowed-tools: [Read, Grep, Glob]
argument-hint: "<directory to analyze>"
---

Analyze the codebase in $ARGUMENTS and return a concise structured summary:
1. Entry points and module map
2. Key dependencies
3. Architecture patterns in use
4. High-risk areas (complexity, missing tests)

Return the summary only — not raw file contents.
```

The hint ensures `$ARGUMENTS` is always a real path before the skill runs.

---

**When to use `argument-hint` vs passing arguments inline**

| Invocation | What happens |
|---|---|
| `/analyze-codebase src/payments/` | `$ARGUMENTS = "src/payments/"` — hint is skipped, skill runs immediately |
| `/analyze-codebase` (no hint configured) | `$ARGUMENTS = ""` — skill runs against nothing |
| `/analyze-codebase` (hint configured) | Claude Code prompts with hint text, waits for input, then runs with the provided value |

Use `argument-hint` on any skill where omitting the argument would produce meaningless output. Skills that operate on the current working directory or the current file may not need it.

In [ ]:
# Demonstrating context isolation: verbose sub-task runs in separate context;
# only its summary enters the main session. This is the principle behind
# context: fork in SKILL.md.

def run_skill_without_fork(main_task: str, skill_task: str) -> dict:
    """
    WITHOUT context: fork:
    Skill output accumulates in the main session's message history.
    Subsequent main task work has less context budget.
    """
    messages = []

    # Skill runs inline — verbose output enters main context
    skill_response = client.messages.create(
        model=MODEL, max_tokens=512,
        messages=[{"role": "user", "content": skill_task}]
    )
    skill_output = skill_response.content[0].text

    # Skill output is now IN the main session messages
    messages.append({"role": "user", "content": skill_task})
    messages.append({"role": "assistant", "content": skill_output})
    messages.append({"role": "user", "content": main_task})

    main_response = client.messages.create(
        model=MODEL, max_tokens=256, messages=messages
    )
    return {
        "pattern": "without fork",
        "context_messages": len(messages),
        "skill_output_in_context": True,
        "skill_token_estimate": len(skill_output.split()),
        "main_response": main_response.content[0].text[:100]
    }


def run_skill_with_fork(main_task: str, skill_task: str) -> dict:
    """
    WITH context: fork:
    Skill runs in an isolated context. Only its summary enters the main session.
    Main session context stays clean.
    """
    # Skill runs in isolation — we extract only the summary
    skill_response = client.messages.create(
        model=MODEL, max_tokens=512,
        system="You are running in an isolated sub-agent context. Return only a concise summary, not the full exploration output.",
        messages=[{"role": "user", "content": skill_task}]
    )
    # Only the summary enters the main session — not the full exploration
    skill_summary = skill_response.content[0].text

    # Main session only sees the compressed summary
    messages = [
        {"role": "user", "content": f"[Skill summary]:\n{skill_summary}\n\n{main_task}"}
    ]

    main_response = client.messages.create(
        model=MODEL, max_tokens=256, messages=messages
    )
    return {
        "pattern": "with fork",
        "context_messages": len(messages),
        "skill_output_in_context": False,
        "skill_token_estimate": len(skill_summary.split()),
        "main_response": main_response.content[0].text[:100]
    }


SKILL_TASK = "Analyze the architecture of a payment processing system with 20 modules, tracing all dependencies."
MAIN_TASK = "Based on the analysis, what are the top 3 refactoring priorities?"

print("Running skill comparison...")
result_no_fork = run_skill_without_fork(MAIN_TASK, SKILL_TASK)
result_fork = run_skill_with_fork(MAIN_TASK, SKILL_TASK)

print("\n=== Without context: fork ===")
for k, v in result_no_fork.items():
    print(f"  {k}: {v}")

print("\n=== With context: fork ===")
for k, v in result_fork.items():
    print(f"  {k}: {v}")

print("\nOBSERVE: With fork, the main session receives a summary, not the full exploration.")
print("This preserves context budget for subsequent work.")

**Key exam facts for 3.2:**
- Commands: `.claude/commands/` (project/shared) vs `~/.claude/commands/` (personal)
- Skills: `.claude/skills/<name>/SKILL.md` with YAML frontmatter
- `context: fork` — skill runs in isolated sub-agent; prevents verbose output polluting main session
- `allowed-tools` — restricts which tools the skill can call (e.g., read-only during analysis)
- `argument-hint` — prompts developer for parameters when skill is invoked without arguments
- Skills = on-demand invocation. CLAUDE.md = always-loaded universal standards. These serve different purposes.
- Personal skill variants: create in `~/.claude/skills/` with a different name to avoid affecting teammates

---
## Task Statement 3.3: Apply path-specific rules for conditional convention loading

Not all conventions apply everywhere. Test file standards are irrelevant when editing a migration script; Terraform rules shouldn't load when writing a React component. Path-specific rules in `.claude/rules/` solve this by tying standards to the files they actually govern — so only relevant context loads for whatever is being edited, and the rest stays out of the token budget.

**What this means in practice:** Files in `.claude/rules/` can have YAML frontmatter with a `paths` field containing glob patterns. When Claude Code is editing a file that matches the pattern, that rule file loads. When it's editing a non-matching file, it doesn't. This means test conventions only load when editing test files, Terraform rules only load when editing `.tf` files, and so on.

**Why it matters for an architect:** The advantage over directory-level CLAUDE.md files is that path-scoped rules can follow file *type* regardless of directory location. Test files in a project are typically spread across many directories alongside the code they test (e.g., `Button.test.tsx` next to `Button.tsx`). A directory CLAUDE.md can't reach across directories — a glob pattern can.

**Core concepts:**
- `.claude/rules/<name>.md` files with YAML frontmatter `paths:` field containing glob patterns (e.g., `["**/*.test.tsx", "tests/**/*.py"]`)
- Rules load ONLY when Claude Code is editing a file that matches the glob — irrelevant conventions stay out of context and don't consume token budget
- Glob patterns target file types across the entire codebase regardless of directory depth

**Key risk:** Directory-level CLAUDE.md files cannot follow test files spread throughout the codebase alongside the code they test. When conventions apply to a file *type* (e.g., all test files, all Terraform configs), use `.claude/rules/` with glob patterns — not a directory CLAUDE.md.

### `.claude/rules/` File Structure

Each file in `.claude/rules/` is a Markdown file with YAML frontmatter. The `paths:` field lists glob patterns. A rule loads **only** when Claude Code is actively editing a file that matches at least one pattern.

```
.claude/
  rules/
    testing.md          # loads for *.test.tsx, *.test.ts, tests/**/*.py
    api-handlers.md     # loads for src/routes/**/*.py
    terraform.md        # loads for terraform/**/*.tf
```

---

**Example — `.claude/rules/testing.md`**

```yaml
---
paths:
  - "**/*.test.tsx"
  - "**/*.test.ts"
  - "tests/**/*.py"
---
```
```markdown
# Test Conventions
- Each test must have a descriptive name starting with `test_` (Python) or `it should` (TS)
- Mock all network calls — never make real HTTP requests in tests
- Use shared fixtures from tests/conftest.py for database setup
- Cover: happy path, edge cases, error conditions
```

---

**Example — `.claude/rules/api-handlers.md`**

```yaml
---
paths:
  - "src/routes/**/*.py"
  - "src/api/**/*.py"
---
```
```markdown
# API Handler Conventions
- All handlers must be async
- Validate input with Pydantic before processing
- Return {data, error} envelope
- Log all exceptions with user_id and request_id context
```

---

Rules load **only** when the active file matches. Editing `src/utils/formatting.py` loads neither rule above — only the root `CLAUDE.md` applies.


### Directory `CLAUDE.md` vs Path-Scoped Rules

| | Directory `CLAUDE.md` | `.claude/rules/` with glob patterns |
|---|---|---|
| **When it applies** | When editing any file inside that directory or a subdirectory | When editing any file whose path matches the glob, anywhere in the repo |
| **Scope** | Tied to a single directory subtree | Follows file type across the entire codebase |
| **Limitation** | Cannot span multiple directories; a single `CLAUDE.md` can't reach `src/components/Button.test.tsx` and `src/api/payments.test.ts` simultaneously | Requires understanding glob syntax; slightly more setup |
| **Best for** | Conventions that truly belong to one subsystem (e.g., a specific microservice) | Conventions tied to file type — test files, Terraform configs, API handlers spread across many directories |

---

**When path-scoped rules win:** Test files in most projects live alongside the code they test — `Button.test.tsx` next to `Button.tsx`, `payments.test.ts` next to `payments.ts`. A directory `CLAUDE.md` can only cover one folder; `paths: ["**/*.test.*"]` covers every test file in the repo with one rule.


**Key exam facts for 3.3:**
- `.claude/rules/` files with YAML frontmatter `paths:` field + glob patterns
- Rules load ONLY when editing a file that matches the glob — reduces irrelevant context and token usage
- Path-scoped rules beat directory CLAUDE.md when conventions span multiple directories (e.g., test files throughout the codebase)
- `paths: ["**/*.test.tsx"]` matches all test files regardless of directory location

---
## Task Statement 3.4: Determine when to use plan mode vs direct execution

Jumping straight into making changes is fine for a well-scoped bug fix with a clear stack trace. It becomes expensive for a library migration that touches 45 files, because wrong early decisions compound — each file Claude edits may lock in an approach that the next file contradicts. Plan mode separates investigation from implementation so the approach is validated before any changes are made.

**What this means in practice:** Plan mode is Claude Code's investigation-before-action mode. The agent reads, explores, and produces a plan without making changes. Direct execution makes changes immediately. The decision rule is straightforward: if the task involves architectural decisions, multiple valid approaches, or changes across many files, use plan mode first. If the task is a well-understood single-file change with a clear scope, direct execution is fine.

**Why it matters for an architect:** The cost of not using plan mode on a complex task is rework. If Claude Code starts restructuring 45 files without understanding dependencies first, it may make decisions early in the process that require undoing later. Plan mode is cheap insurance against that. The Explore subagent is a related concept — it isolates verbose discovery output so it doesn't exhaust the main context window during multi-phase tasks.

**Core concepts:**
- Plan mode: Claude Code investigates, explores, and produces a numbered plan without making any changes — the developer reviews and approves before execution begins
- Direct execution: changes are made immediately with no approval step
- Explore subagent: isolates verbose discovery output from the main session context window; returns only a summary to the main agent
- Decision signals for plan mode: architectural decisions, 10+ files affected, multiple valid approaches, changes expensive to reverse, open-ended scope requiring discovery first

**Anti-patterns to avoid:**
- Using direct execution for architectural changes or library migrations — rework is expensive when the first approach turns out to be wrong
- Using plan mode for a single-file, stack-trace-clear bug fix — unnecessary overhead with no benefit
- Treating plan mode as a guarantee of no rework — it surfaces assumptions upfront but doesn't eliminate the possibility of adjustments during execution

### Plan Mode vs Direct Execution Decision Table

| Task | Mode | Key signals |
|------|------|-------------|
| Fix null pointer bug in `src/payments/refund.py` — stack trace points to line 47 | **Direct execution** | Single file, clear location, no ambiguity about approach |
| Add a date validation check to the registration form handler | **Direct execution** | Narrow scope, one function, clear requirement |
| Restructure the monolithic payment service into microservices | **Plan mode** | Architectural decision, dozens of files, multiple valid approaches, expensive to reverse |
| Migrate from `requests` to `httpx` across the entire codebase (45+ files) | **Plan mode** | 45+ files, async vs sync choices have cascading implications, need to understand all usage patterns first |
| Choose between integrating Stripe or PayPal for payment processing | **Plan mode** | Multiple valid approaches with different infrastructure requirements; explore tradeoffs before committing |
| Add comprehensive tests to a legacy codebase with no existing tests | **Plan mode** | Open-ended scope, unknown structure, must map the codebase first to identify high-impact areas |

**Decision rule:** If the task involves architectural decisions, multiple valid approaches, or changes across many files — use plan mode. If it is a well-understood single-file change with a clear scope — use direct execution.


In [ ]:
# Two-phase workflow: investigate and plan first, then implement.
# Phase 1 produces an explicit plan for review before any changes are made.

def plan_then_execute(task: str, codebase_context: str) -> dict:
    """
    Phase 1 (plan mode equivalent): Explore and plan without making changes.
    Phase 2 (direct execution): Implement the approved plan.

    """
    print("Phase 1: Investigation and planning (no changes)")

    plan_response = client.messages.create(
        model=MODEL,
        max_tokens=512,
        system=(
            "You are in PLAN MODE. Your job is to analyze the task and produce a step-by-step "
            "implementation plan. Do NOT make any changes. Do NOT write code. "
            "Only analyze, identify risks, and produce a numbered plan."
        ),
        messages=[{"role": "user", "content": f"Codebase context:\n{codebase_context}\n\nTask: {task}"}]
    )
    plan = plan_response.content[0].text
    print(f"  Plan generated ({len(plan.split())} words)")
    print(f"  [Developer reviews and approves plan before execution]")

    print("\nPhase 2: Execution based on approved plan")
    exec_response = client.messages.create(
        model=MODEL,
        max_tokens=512,
        system="You are implementing an approved plan. Be specific and precise. Focus on Step 1 only.",
        messages=[{
            "role": "user",
            "content": f"Approved plan:\n{plan}\n\nImplement Step 1 of this plan for the following codebase:\n{codebase_context}"
        }]
    )

    return {
        "plan": plan,
        "step_1_implementation": exec_response.content[0].text
    }


CODEBASE_CONTEXT = """
Payment service, Python/FastAPI. Current structure:
- src/main.py: entry point, registers all routes
- src/routes/: payment.py, refund.py, subscription.py
- src/services/: payment_service.py (1200 lines, does everything)
- src/db/: repository.py, models.py
- tests/: 12 test files, 40% coverage
"""

result = plan_then_execute(
    task="Break payment_service.py into focused service classes",
    codebase_context=CODEBASE_CONTEXT
)

print(f"\n=== Plan ===")
print(result["plan"])
print(f"\n=== Step 1 Implementation ===")
print(result["step_1_implementation"])

**Key exam facts for 3.4:**

**Use plan mode when:** architectural decisions, multiple valid approaches, 10+ files affected, changes are expensive to reverse, open-ended scope requiring discovery first.

**Use direct execution when:** single file, clear stack trace, well-understood scope, no architectural decision involved.

**Explore subagent:** use during verbose discovery phases to prevent context window exhaustion — returns a summary, not the full exploration output.

**Combined pattern:** plan mode for investigation + direct execution for implementation (e.g., plan a library migration, then execute the planned approach).

---
## Task Statement 3.5: Apply iterative refinement techniques for progressive improvement

The gap between a task description and the output you actually wanted is almost always a specification problem, not a model capability problem. Vague instructions produce vague results; a concrete before/after example removes the ambiguity that natural language leaves unresolved. This task statement covers four techniques for narrowing that gap — each one targeting a different category of underspecification that causes inconsistent or incorrect output.

**What this means in practice:** Four techniques for getting better output through iteration:
1. **Concrete input/output examples** — when prose descriptions produce inconsistent results, show 2-3 examples of input and expected output
2. **Test-driven iteration** — write tests first, share failures to guide fixes
3. **Interview pattern** — have Claude ask questions before implementing (surfaces design considerations you hadn't anticipated)
4. **Batch vs sequential issue reporting** — provide all interacting issues in one message; report independent issues sequentially

**Why it matters for an architect:** Natural language task descriptions are ambiguous. "Reformat the data" means different things to different people. A concrete before/after example removes the ambiguity. Test-driven iteration removes the subjective judgment call about whether the output is correct — either the tests pass or they don't.

**Core concepts:**
- Four iterative refinement techniques: concrete input/output examples, test-driven iteration (write tests first; share failures to drive fixes), interview pattern (Claude asks clarifying questions before implementing), batch vs. sequential issue reporting
- Concrete examples are most effective when prose descriptions produce inconsistent results — 2-3 examples including edge cases and at least one "acceptable pattern — do not flag" case
- Interview pattern surfaces design decisions (cache invalidation strategy, failure modes, sync vs. async tradeoffs) before implementation commits to an approach
- Interacting issues (fixes may affect each other) must be reported together; independent issues should be fixed sequentially

**Anti-patterns to avoid:**
- Providing only one example — insufficient for generalization to cases the prompt author didn't anticipate
- Batching independent issues in one message — adds unnecessary context and complicates each fix
- Fixing interacting issues sequentially — later fixes may conflict with or undo earlier ones
- Describing transformation rules in prose when a before/after example would be unambiguous

### Technique 1: Concrete Input/Output Examples

Use when prose descriptions produce inconsistent results. 2–3 examples with edge cases remove ambiguity that natural language cannot.

---

**Anti-pattern — prose description:**

```
Write a function that reformats user records from the legacy format to the new format.
Clean up the date fields and normalize the name.
```

The model must guess: which fields? what date format? title case or upper? what does "normalize" mean?

---

**Correct pattern — concrete examples:**

```
Write a function that reformats user records from the legacy format to the new format.

Example 1:
Input:  {"FIRST_NM": "jane", "LAST_NM": "smith", "DOB": "19850312", "ACCT_STAT": "A"}
Output: {"full_name": "Jane Smith", "date_of_birth": "1985-03-12", "status": "active"}

Example 2:
Input:  {"FIRST_NM": "BOB", "LAST_NM": "jones", "DOB": "19901225", "ACCT_STAT": "I"}
Output: {"full_name": "Bob Jones", "date_of_birth": "1990-12-25", "status": "inactive"}

Example 3 (edge case — missing DOB):
Input:  {"FIRST_NM": "alice", "LAST_NM": "wong", "DOB": None, "ACCT_STAT": "A"}
Output: {"full_name": "Alice Wong", "date_of_birth": None, "status": "active"}
```

Three examples make explicit: the exact field mapping (`FIRST_NM` → `full_name`), the date format (`YYYYMMDD` → `YYYY-MM-DD`), the status codes (`A` → `active`), and null handling. No prose description achieves this unambiguously.

**Include at least one edge case** — the model generalizes from examples, and a missing edge case means it will invent behavior for inputs you didn't anticipate.

### Technique 2: Test-Driven Iteration

Write the tests first. Share the specific failure output. The failure is an unambiguous specification — no prose description of "what's wrong" is needed.

---

**Step 1 — Ask for an implementation:**

```
Write a reformat_user function that converts legacy user records to the new format.
Use the input/output contract from the examples above.
```

---

**Step 2 — Run the tests, get a failure, share it verbatim:**

```
The implementation has a bug. Here is the test failure:

FAILED tests/test_reformat.py::test_null_dob

    def test_null_dob():
        record = {"FIRST_NM": "alice", "LAST_NM": "wong", "DOB": None, "ACCT_STAT": "A"}
        result = reformat_user(record)
>       assert result["date_of_birth"] is None

TypeError: 'NoneType' object is not subscriptable
    in dob = f"{record['DOB'][:4]}-..."

1 failed, 4 passed

Fix the implementation so all tests pass. Change only what is needed.
```

---

The failure pinpoints the exact line, the exact input that triggers it, and the expected vs. actual behavior. "Change only what is needed" prevents unnecessary rewrites.

**Why this beats describing the bug in prose:** "The function crashes on null DOB" requires the model to locate the bug. The stack trace shows exactly where the crash happens — it's a more precise specification than any natural language description.

### Technique 3: Interview Pattern

Ask Claude to question you before implementing. It will surface design decisions that prose task descriptions leave implicit — decisions that would require rework if assumed incorrectly.

---

**Prompt:**

```
I need to add a caching layer to the payment service's database queries.

Before implementing, ask me 3–5 clarifying questions that will affect the design.
Focus on questions where my answer would change the implementation approach.
Do not implement anything yet.
```

---

**The questions Claude asks (representative):**

```
1. Should the cache be local (in-process, per-instance) or distributed (Redis/Memcached shared
   across all service instances)?

2. What is the cache invalidation strategy when an order is updated or a payment status changes?
   Should writes invalidate the cache immediately, or rely on TTL expiry?

3. Are there query results that must never be cached — for example, balance checks before
   a payment authorization, where stale data could cause an overdraft?

4. Should cache misses be transparent to the caller (cache-aside pattern), or should the
   service be aware of the cache and populate it explicitly?

5. Do you need cache hit/miss metrics? If so, where should they be emitted — existing
   observability infrastructure or a new counter?
```

---

Each question represents a design decision where the answer changes what gets built. "Distributed vs. local" changes the dependency stack and deployment config. "Invalidation strategy" changes the data access layer. "Never cache balance checks" changes which methods even participate in caching.

**When to use it:** any task with non-obvious architectural decisions — caching, queuing, auth, retry logic, data modeling. The interview pattern surfaces the decision tree before code is written rather than after.

### Technique 4: Batch vs Sequential Issue Reporting

**Interacting issues** must go in one message — later fixes may conflict with or depend on earlier ones. **Independent issues** should be reported one at a time — batching them adds context noise without benefit.

---

**Interacting issues — one message:**

```
There are three issues with the refund function that interact with each other:

1. The function doesn't validate that `amount` is positive before processing.
2. The function doesn't check that the order is within the 30-day return window
   before calling the refund API.
3. The function doesn't roll back the database record if the refund API call fails.

Fix all three together. The correct order is: validate amount → check return window →
call API → roll back on failure. A fix to #3 must account for the guard added by #2
(don't call the API if the order is ineligible — there's nothing to roll back).
```

Fixing these sequentially risks inconsistency: fix #3 written before fix #2 exists won't know to skip the rollback on ineligible orders.

---

**Independent issues — separate messages:**

Message 1:
```
Fix the typo on line 47 of src/refund.py: change "recieve" to "receive".
```

Message 2 (after fix 1 is applied):
```
Add a docstring to the calculate_tax function in src/tax.py.
```

Message 3 (after fix 2 is applied):
```
Update the Python version requirement in README.md from 3.9 to 3.11.
```

Each fix is isolated. Sending all three in one message adds irrelevant context to each fix and makes it harder to review the change for each issue independently.

---

**Decision rule:**

| Situation | Approach |
|---|---|
| Fix A changes code that Fix B also touches | Batch in one message |
| Fix A's guard condition affects whether Fix B should run | Batch in one message |
| Fixes are in different files with no shared logic | Sequential, one message each |
| One is a typo/rename, the other is a logic change | Sequential |

**Key exam facts for 3.5:**
- **Concrete examples** are the most effective technique when prose produces inconsistent results — 2-3 input/output examples, including edge cases
- **Test-driven iteration:** tests first → share specific failures → targeted fix
- **Interview pattern:** Claude asks questions before implementing — surfaces cache invalidation strategies, failure modes, design tradeoffs
- **Batch interacting issues** in one message. **Fix independent issues** sequentially.
- Concrete examples beat detailed prose instructions for transformation tasks

---
## Task Statement 3.6: Integrate Claude Code into CI/CD pipelines

Claude Code is designed for interactive developer sessions, but many of its most valuable use cases — automated code review on every PR, test generation in a pre-merge gate, security scanning before deployment — require it to run unattended inside a pipeline. This task statement covers how to bridge that gap: the flags that make Claude Code non-interactive, the output formats that make its findings machine-parseable, and the architectural patterns that make automated reviews trustworthy.

**What this means in practice:** Claude Code runs interactively by default — it waits for user input. In CI/CD, there's no user, so the pipeline hangs. The `-p` (or `--print`) flag switches Claude Code to non-interactive mode: it processes the prompt, prints output to stdout, and exits. For machine-parseable output (inline PR comments, structured findings), use `--output-format json` with `--json-schema`. CLAUDE.md provides project context (testing standards, review criteria) to the CI-invoked instance.

**Why it matters for an architect:** Session context isolation is the key architectural insight. The Claude session that generated code retains its reasoning from generation — it's less likely to question its own decisions when reviewing. An independent review instance (a fresh Claude invocation) catches issues the generator would rationalize away. This is why the correct pattern is two separate Claude invocations: one to generate, one to review.

**Core concepts:**
- `-p` / `--print` flag switches Claude Code to non-interactive mode: process prompt, print to stdout, exit — required for all CI/CD usage; without it the pipeline hangs waiting for input
- `--output-format json` with `--json-schema` produces machine-parseable structured findings for downstream pipeline steps (inline PR comments, automated routing)
- CLAUDE.md provides project context (testing standards, review criteria) to CI-invoked instances
- Session context isolation: the session that generated code retains its reasoning and rationalizes its own decisions; an independent review invocation (fresh context, no generation history) is more effective
- Incremental reviews: include prior findings in context and instruct to report only new or still-unaddressed issues

**Key risks:**
- `CLAUDE_HEADLESS=true` and `--batch` are not real flags — the only correct non-interactive invocation is `claude -p "<prompt>"`
- The code-generating session is less effective at reviewing its own output — always use a separate review invocation

### Claude Code CI/CD Flags Reference

| Flag | Purpose | When to use | Example command | Wrong alternative |
|------|---------|-------------|-----------------|-------------------|
| `-p` / `--print` | Non-interactive mode: process prompt, print to stdout, exit | **Always in CI/CD** — without it, Claude Code waits for user input and the pipeline hangs | `claude -p "Review this PR for security issues"` | `claude "Review this PR" < /dev/null` (workaround, not correct) |
| `--output-format json` | Output structured JSON instead of prose | When downstream tools need to parse the output (e.g., posting inline PR comments) | `claude -p "Review for security" --output-format json --json-schema review_schema.json` | Parsing prose output with regex — brittle and breaks on format changes |
| `--json-schema <file>` | Enforce a specific JSON schema for the output | When you need guaranteed schema compliance for downstream processing | `--json-schema .claude/schemas/review-finding.json` | Asking Claude to "return JSON" in the prompt — non-deterministic |

---

> **CRITICAL:** `CLAUDE_HEADLESS=true` and `--batch` are **not** real flags. The only correct non-interactive invocation is `claude -p "<prompt>"`.


### CI Review Schema

Use `--json-schema` in CI to enforce this structure for downstream parsing. Contrast with asking Claude to "return JSON" in the prompt — that relies on the model following prose instructions and is non-deterministic.

```json
{
  "type": "object",
  "properties": {
    "findings": {
      "type": "array",
      "items": {
        "type": "object",
        "properties": {
          "file":              { "type": "string" },
          "line":              { "type": "integer" },
          "severity":          { "type": "string", "enum": ["critical", "high", "medium", "low"] },
          "category":          { "type": "string", "enum": ["security", "bug", "performance", "style"] },
          "description":       { "type": "string" },
          "suggested_fix":     { "type": "string" },
          "detected_pattern":  { "type": "string" }
        },
        "required": ["file", "line", "severity", "category", "description"]
      }
    },
    "summary":        { "type": "string" },
    "total_findings": { "type": "integer" }
  },
  "required": ["findings", "summary", "total_findings"]
}
```

With `--json-schema`, Claude Code guarantees the output matches this schema. The CI pipeline can safely index `findings[i].severity` and `findings[i].file` to post inline PR comments without defensive parsing.


In [11]:
# Session context isolation: independent review instance catches more issues
# The generator retains its reasoning context — it rationalizes its own decisions.
# An independent reviewer has no such bias.

CODE_TO_REVIEW = """
def transfer_funds(from_account: str, to_account: str, amount: float) -> dict:
    # Deduct from source
    db.execute(f"UPDATE accounts SET balance = balance - {amount} WHERE id = '{from_account}'")
    # Add to destination
    db.execute(f"UPDATE accounts SET balance = balance + {amount} WHERE id = '{to_account}'")
    return {"status": "success", "amount": amount}
"""

def self_review(code: str, generator_reasoning: str) -> str:
    """Simulates self-review: same session that generated the code reviews it.
    The generator's reasoning context is still active — it tends to justify its decisions."""
    response = client.messages.create(
        model=MODEL, max_tokens=384,
        system=(
            f"You just wrote this code with the following reasoning: {generator_reasoning}. "
            "Now review it for issues."
        ),
        messages=[{"role": "user", "content": f"Review this code:\n{code}"}]
    )
    return response.content[0].text


def independent_review(code: str) -> str:
    """Simulates independent review: fresh instance with no generation context.
    No prior reasoning to defend — more likely to catch issues."""
    response = client.messages.create(
        model=MODEL, max_tokens=384,
        system="You are a security-focused code reviewer. You have no context about how this code was written.",
        messages=[{"role": "user", "content": f"Review this code for security issues and bugs:\n{code}"}]
    )
    return response.content[0].text


GENERATOR_REASONING = (
    "I used f-strings for the SQL queries because it's simpler and the account IDs "
    "come from our internal system so they're trusted. I chose not to use a transaction "
    "because the operations are fast and failures are rare."
)

print("=== Self-review (same session) ===")
self_result = self_review(CODE_TO_REVIEW, GENERATOR_REASONING)
print(self_result)

print("\n=== Independent review (fresh instance) ===")
independent_result = independent_review(CODE_TO_REVIEW)
print(independent_result)

print("\nOBSERVE: The independent reviewer is more likely to flag:")
print("  - SQL injection (f-strings with account IDs)")
print("  - Missing transaction (partial failure leaves accounts in inconsistent state)")
print("  - No amount validation (negative amounts, zero)")
print("The self-reviewer may rationalize these away based on its generation reasoning.")

=== Self-review (same session) ===
## Code Review

Your original reasoning contains several flawed assumptions. Here's a detailed breakdown:

---

### 🚨 Critical Issues

#### 1. SQL Injection (Your "trusted inputs" reasoning is flawed)

```python
# Your assumption: "account IDs come from our internal system so they're trusted"
# Reality: Defense in depth - you can't always guarantee call origin

# Malicious from_account: "' OR '1'='1"
# Malicious amount: "1 OR 1=1 --"
f"UPDATE accounts SET balance = balance - {amount} WHERE id = '{from_account}'"
```

**Fix:** Always use parameterized queries:
```python
db.execute(
    "UPDATE accounts SET balance = balance - ? WHERE id = ?",
    (amount, from_account)
)
```

---

#### 2. No Transaction (Your "failures are rare" reasoning is wrong)

```python
# If this succeeds...
db.execute(f"UPDATE accounts SET balance = balance - {amount}...")
# ...and THIS fails, money vanishes
db.execute(f"UPDATE accounts SET balance = balance + {amount}...")
```


In [12]:
# Incremental review: avoid duplicate comments on re-runs
# When a new commit is pushed, include prior review findings
# and instruct Claude to report only NEW or STILL-UNADDRESSED issues.

PRIOR_REVIEW_FINDINGS = [
    {"file": "src/refund.py", "line": 12, "severity": "critical",
     "description": "SQL injection via f-string interpolation of order_id",
     "status": "addressed"},  # Developer fixed this
    {"file": "src/refund.py", "line": 18, "severity": "high",
     "description": "No error handling on charge_api.refund() call",
     "status": "unaddressed"},  # Still present
]

NEW_DIFF = """
# Developer fixed SQL injection but introduced a new issue
- query = f"SELECT * FROM orders WHERE id = {order_id}"
+ query = "SELECT * FROM orders WHERE id = %s"
+ db.execute(query, (order_id,))  # Fixed SQL injection
+ logging.debug(f"Refund requested for order {order_id} by user {user_id} amount {amount}")  # PII in logs
"""

incremental_review_prompt = f"""
Review the new commit diff below.

Prior review findings (DO NOT re-report these):
{json.dumps(PRIOR_REVIEW_FINDINGS, indent=2)}

Report ONLY:
1. Issues that are still unaddressed from the prior review
2. New issues introduced by this commit

Do NOT re-report issues marked as 'addressed'.

New diff:
{NEW_DIFF}
"""

incremental_response = client.messages.create(
    model=MODEL, max_tokens=384,
    messages=[{"role": "user", "content": incremental_review_prompt}]
)

print("=== Incremental CI review (avoids duplicate comments) ===")
print(incremental_response.content[0].text)

=== Incremental CI review (avoids duplicate comments) ===
## Review Findings

### Unaddressed Issues from Prior Review

| File | Line | Severity | Description |
|------|------|----------|-------------|
| `src/refund.py` | 18 | **High** | No error handling on `charge_api.refund()` call — still not addressed in this diff |

---

### New Issues Introduced by This Commit

| File | Severity | Description |
|------|----------|-------------|
| `src/refund.py` | **High** | **PII/sensitive data written to logs** — the new `logging.debug(...)` line records `user_id` and `amount`, which are personally identifiable and financially sensitive. Log files are often stored long-term, shipped to third-party aggregators, or accessible to personnel who should not see this data. This may violate GDPR, PCI-DSS, or similar compliance requirements. |

**Recommendation:** Remove `user_id` and `amount` from the log statement, or replace with non-sensitive identifiers (e.g., a masked/tokenized reference). If thi

**Key exam facts for 3.6:**
- `-p` / `--print` flag is the ONLY correct way to run Claude Code non-interactively in CI. `CLAUDE_HEADLESS`, `--batch` are not real flags.
- `--output-format json` + `--json-schema` for machine-parseable structured findings
- CLAUDE.md provides project context (testing standards, review criteria) to CI-invoked instances
- **Session context isolation:** the session that generated code is less effective at reviewing it — use an independent review instance
- Incremental reviews: include prior findings + instruct to report only new/unaddressed issues
- Provide existing test files in context so test generation avoids duplicating existing coverage

---
## Domain 3 Capstone Project: Team-Ready Claude Code Setup

**Description:** Design a complete Claude Code configuration for a real project — demonstrating all six task statements. This is a configuration + API exercise.

**Deliverables:**
1. Project-level CLAUDE.md with `@import` structure (3.1)
2. Shared `/review` command (3.2)
3. Codebase analysis skill with `context: fork` (3.2)
4. Path-scoped rules for test files and API handlers (3.3)
5. CI review pipeline using structured JSON output (3.6)

### Domain 3 Capstone: Complete Project Configuration

All six configuration files for a team-ready Claude Code setup.

---

#### `.claude/CLAUDE.md`

```markdown
# Payment Service — Claude Code Configuration
# Project-level: committed to version control, applies to all team members

## Architecture
FastAPI + PostgreSQL. Entry point: src/main.py.
Repository pattern for DB access. All business logic in src/services/.

## Standards (modular — loaded via @import)
@import .claude/standards/testing.md
@import .claude/standards/api-conventions.md
@import .claude/standards/security.md

## Non-negotiable
- Run `pytest tests/ -v` before marking any task complete
- Never commit secrets or credentials
- All public functions require type hints and docstrings
```

---

#### `.claude/commands/review.md`

```markdown
Review current changes for:
1. Security: SQL injection, XSS, hardcoded credentials, insecure deserialization
2. Correctness: Logic errors, missing error handling, race conditions
3. Testing: New code has corresponding tests with edge case coverage

For each issue: file, line, severity (critical|high|medium|low), description, fix.
Skip style preferences. Skip subjective improvements.
```

---

#### `.claude/skills/analyze-codebase/SKILL.md`

```markdown
---
context: fork
allowed-tools: [Read, Grep, Glob]
argument-hint: "<directory or module to analyze>"
---

Analyze $ARGUMENTS and return a concise structured summary:
1. Entry points and module map
2. Key dependencies
3. Architecture patterns
4. High-risk areas (complexity, missing tests)

Return the summary only — not raw file contents.
```

---

#### `.claude/rules/testing.md`

```markdown
---
paths:
  - "**/*.test.tsx"
  - "**/*.test.ts"
  - "tests/**/*.py"
---

# Test Conventions
- pytest for Python, Vitest for TypeScript
- Mock all external services — no real network calls
- Use fixtures from tests/conftest.py
- Cover: happy path, validation errors, service failures
```

---

#### `.claude/rules/api-handlers.md`

```markdown
---
paths:
  - "src/routes/**/*.py"
  - "src/api/**/*.py"
---

# API Handler Conventions
- All handlers async
- Validate input with Pydantic before processing
- Return {data, error} envelope
- Log exceptions with user_id and request_id context
```

---

#### `.mcp.json`

```json
{
  "mcpServers": {
    "github": {
      "command": "npx",
      "args": ["-y", "@modelcontextprotocol/server-github"],
      "env": {
        "GITHUB_TOKEN": "${GITHUB_TOKEN}"
      }
    }
  }
}
```


In [13]:
# Capstone: CI pipeline simulation
# Models the full review pipeline: structured output -> parseable findings

CI_REVIEW_SCHEMA = {
    "type": "object",
    "properties": {
        "findings": {
            "type": "array",
            "items": {
                "properties": {
                    "file": {"type": "string"},
                    "line": {"type": "integer"},
                    "severity": {"type": "string"},
                    "description": {"type": "string"},
                    "detected_pattern": {"type": "string"}
                }
            }
        },
        "new_issues_only": {"type": "boolean"},
        "commit_sha": {"type": "string"}
    }
}

PR_CODE = """
# New code in src/routes/refund.py
async def create_refund(request: Request):
    data = await request.json()  # No Pydantic validation
    order_id = data['order_id']
    amount = data['amount']
    result = refund_service.process(order_id, amount)  # No error handling
    return result  # Raw object, not {data, error} envelope
"""

ci_prompt = f"""
You are running as a CI code reviewer (non-interactive mode, equivalent to `claude -p`).
Review the following new code against the project standards.

Project standards (from CLAUDE.md):
- All API handlers must validate input with Pydantic
- All handlers must be async
- Return {{data, error}} envelope
- Log exceptions with context

Return ONLY valid JSON. No explanation. No markdown.
Schema: {json.dumps(CI_REVIEW_SCHEMA)}

Code to review:
{PR_CODE}

Set commit_sha to "abc123" and new_issues_only to true.
"""

ci_result = client.messages.create(
    model=MODEL, max_tokens=512,
    messages=[{"role": "user", "content": ci_prompt}]
)

raw = ci_result.content[0].text.strip().replace("```json", "").replace("```", "").strip()
print("=== CI Pipeline: Structured Review Output ===")
try:
    parsed = json.loads(raw)
    print(json.dumps(parsed, indent=2))
    print(f"\nFindings: {len(parsed.get('findings', []))}")
    print("This output can be directly posted as inline PR comments.")
except json.JSONDecodeError:
    print(raw)

=== CI Pipeline: Structured Review Output ===
{
  "findings": [
    {
      "file": "src/routes/refund.py",
      "line": 2,
      "severity": "high",
      "description": "Input is not validated with Pydantic. Raw dict access is used instead of a Pydantic model, violating the project standard requiring all API handlers to validate input with Pydantic.",
      "detected_pattern": "data = await request.json()"
    },
    {
      "file": "src/routes/refund.py",
      "line": 3,
      "severity": "high",
      "description": "Direct dictionary key access without validation. This will raise an unhandled KeyError if 'order_id' is missing from the request body.",
      "detected_pattern": "order_id = data['order_id']"
    },
    {
      "file": "src/routes/refund.py",
      "line": 4,
      "severity": "high",
      "description": "Direct dictionary key access without validation. This will raise an unhandled KeyError if 'amount' is missing from the request body.",
      "detected_pattern": "

---
## Domain 3 Complete

**Summary of key exam facts:**

| Task | Core Principle |
|------|----------------|
| 3.1 | Three levels: user (personal), project (team/version controlled), directory (subtree). `@import` for modularity. `/memory` to diagnose. |
| 3.2 | Commands = reusable prompts. Skills = commands + frontmatter (`context: fork`, `allowed-tools`, `argument-hint`). `context: fork` isolates verbose output from main session. |
| 3.3 | `.claude/rules/` + YAML `paths:` glob patterns = conditional loading. Beats directory CLAUDE.md for cross-directory conventions (test files). |
| 3.4 | Plan mode: architectural decisions, 10+ files, multiple valid approaches, open-ended scope. Direct: single file, clear scope, no ambiguity. |
| 3.5 | Concrete examples > prose for transformations. Test failures drive targeted fixes. Interview pattern surfaces design decisions. Batch interacting issues; sequence independent ones. |
| 3.6 | `-p` flag = non-interactive CI mode. `--output-format json` + `--json-schema` = structured findings. Independent review instance catches more than self-review. Include prior findings on re-runs. |